# Laboratorio 7 - Entrega Parcial
## Task 1: Análisis de DQN y Double DQN para detección de intrusiones

## Task 1.1

### a. Estado continuo de 256 dimensiones

DQN sí es arquitecturalmente apropiado. DQN puede recibir estados continuos; no exige imágenes ni estados discretos. El problema no es la cantidad de variables, sino que el vector esté bien normalizado y que las métricas relevantes estén presentes.

Usaríamos una red totalmente conectada (MLP): entrada de 256 valores, dos o tres capas densas con ReLU y una capa final de 4 salidas. Una CNN no es la primera opción porque el vector no tiene vecindad espacial como una imagen. Si los logs se entregaran como secuencias con orden temporal, se podría añadir una LSTM/GRU o atención antes de la MLP.

### b. Espacio de acciones

DQN y sus variantes son apropiados porque hay solo 4 acciones discretas. La red puede calcular los cuatro valores $Q(s,a)$ en una sola pasada y escoger la mayor. DQN deja de ser práctico cuando las acciones son continuas o muy numerosas.

### c. Intrusiones muy raras y Experience Replay

Sea $p<0.001$ la proporción de transiciones críticas en un buffer de tamaño $N$. Con muestreo uniforme, el número esperado de transiciones críticas en un minibatch de tamaño $B$ es:

$$E[K]=Bp<0.001B.$$

Por ejemplo, con $B=64$, $E[K]<0.064$. Además, la probabilidad de que un minibatch no contenga ninguna intrusión es:

$$P(K=0)=(1-p)^B \approx (0.999)^{64}=0.938.$$

Por tanto, cerca del 94% de los minibatches no enseñan al agente qué hacer ante una intrusión real. El buffer queda dominado por tráfico normal y los gradientes favorecen decisiones frecuentes, como ignorar.

La variante indicada es **Prioritized Experience Replay (PER)** con DQN o Double DQN. PER asigna mayor probabilidad a transiciones con gran error TD, que suelen incluir eventos raros o mal aprendidos. Se deben usar pesos de importancia para reducir el sesgo que introduce el muestreo no uniforme. También conviene guardar y revisar explícitamente suficientes casos críticos; PER no sustituye la calidad ni la cobertura de los datos.

### d. Recompensa y falsos positivos

Si se minimizan solo los falsos positivos, el agente puede aprender a ignorar casi todo. Eso baja las alertas equivocadas, pero aumenta los falsos negativos y deja pasar intrusiones reales.

Una recompensa posible por decisión es:

$$r = 100\,\mathbb{1}[\text{intrusión detectada y contenida}] - 200\,\mathbb{1}[\text{intrusión ignorada}] - 10\,\mathbb{1}[\text{falso positivo}] - c(a).$$

Donde $c(a)$ representa el costo operativo: $c(\text{ignorar})=0$, $c(\text{alertar})=1$, $c(\text{bloquear})=4$ y $c(\text{aislar})=12$. Los componentes son: recompensa por detener una intrusión, castigo por falso negativo, castigo por falso positivo y costo de interrupción de cada acción.

El falso negativo recibe la mayor penalización porque una intrusión no contenida puede causar pérdida de datos o caída del servicio. El falso positivo se penaliza menos, pero lo suficiente para no bloquear tráfico normal. Aislar cuesta más que alertar o bloquear porque puede afectar a usuarios legítimos. Los valores exactos deben calibrarse con costos reales, pruebas históricas y límites de servicio.

## Task 1.2

### a. Sobreestimación

En DQN, el máximo usado en el target elige la acción con mayor estimación ruidosa. Si $\widehat{Q}(s,a)=Q(s,a)+\epsilon_a$, entonces normalmente:

$$E[\max_a \widehat{Q}(s,a)] \geq \max_a Q(s,a).$$

El máximo favorece por azar los errores positivos. Double DQN reduce este efecto al separar la red que elige la acción de la red que la evalúa.

Si se sobreestima *bloquear*, el sistema puede bloquear tráfico legítimo con demasiada frecuencia: interrumpe usuarios, aplicaciones y operaciones. Si se sobreestima *ignorar*, puede dejar pasar señales peligrosas: aumenta el riesgo de intrusiones no contenidas.

### b. Targets de DQN y Double DQN

Para una transición $(s,a,r,s')$, el target de DQN estándar es:

$$y_{DQN}=r+\gamma(1-d)\max_{a'}Q(s',a';w^-).$$

Para Double DQN es:

$$y_{DDQN}=r+\gamma(1-d)Q\left(s',\underset{a'}{\arg\max}\ Q(s',a';w);w^-\right).$$

Aquí $r$ es la recompensa, $\gamma$ el descuento y $d$ vale 1 si el episodio termina. $w$ son los parámetros de la red en línea, que se actualiza con el entrenamiento. $w^-$ son los parámetros de la red objetivo, que se copian desde $w$ cada cierto número de pasos y dan mayor estabilidad.

El único cambio de implementación está en calcular el target: DQN usa el máximo de la red objetivo; Double DQN usa la red en línea para obtener el índice de la mejor acción y la red objetivo para obtener el valor de ese índice. La arquitectura, el buffer, la función de pérdida y la actualización periódica de la red objetivo pueden mantenerse iguales.

### c. Estados donde el problema es más grave

La sobreestimación es más grave en estados ambiguos: tráfico borderline, señales incompletas o actividad que parece normal y maliciosa a la vez. En esos estados, los valores de las acciones son parecidos y el ruido de estimación puede cambiar cuál parece mejor. Eso puede producir bloqueos innecesarios o decisiones de ignorar peligrosas.

En estados claramente maliciosos, aislar o bloquear debería tener un valor mucho mayor que ignorar. Como la separación entre acciones es amplia, un error pequeño tiene menos probabilidad de cambiar la decisión. Double DQN sigue siendo útil, pero su beneficio operativo principal está en las decisiones inciertas.